# <font color='black'> 📌***Entendendo o Problema (Negócio)***

# <font color='Black'>⚙️***Importação e Configurações***

In [ ]:
pd.set_option('display.max_columns', 30)
pd.set_option('display.max_rows', 30)

# <font color='lightgreen'> 🗃️ ***Base de Dados***

# <font color='lightgreen'> 📊 ***Análise Exploratória (EDA)***

# <font color='lightgreen'> 🗑️ ***Data Cleaning***

# <font color='lightgreen'> 🏗️ ***Feature Engineering***

# <font color='lightgreen'> 🤖 ***Machine Learning (Seleção de Modelo)***

***Baseline (CV Simples):*** Use cross_val_score (ex: 5-10 folds) nos 3 modelos com parâmetros padrão. Isso te dá a performance base de cada algoritmo.
1. BaselineTreine 3 modelos (ex: RF, SVM, XGBoost) com parâmetros padrão.Use um cross_val_score simples (ex: 5 folds) apenas para ver o desempenho inicial.
2. Avaliação de cada modelo pelo cross_val_predict (matrix de Confusão, classification_report)

# <font color='lightgreen'> 🎛️ ***Tuning de Hiperparâmetros***

***Ajuste com Nested CV:*** Para os modelos que passaram pelo baseline, rode a Validação Aninhada para otimizar os hiperparâmetros de forma justa.
* 1- Fazer um Random Search CV (60 a 100 iterações -- 99%)
* 2- Fazer outro Random Search CV com os melhores do primeiro (20 a 40 iterações)
* 3- Fazer GridSearch CV para ajuste fino (grade pequena -- ex: 2x2 ou 3x3)

OBS: número "mágico" muito famoso na ciência de dados baseado em um estudo clássico de Bergstra e Bengio: 60 iterações.
*       tem 95% de probabilidade de encontrar uma solução que esteja entre os 5% melhores resultados possíveis de todo o espaço de busca.

In [ ]:
from sklearn.model_selection import RandomizedSearchCV, cross_val_score, KFold

# 1. Configura as divisões
outer_cv = KFold(n_splits=10, shuffle=True, random_state=42)
inner_cv = KFold(n_splits=3, shuffle=True, random_state=42)

grade_de_parametros = {
    'model__n_neighbors': np.linspace(5, 25, 10, dtype=int),
    'model__weights': ['uniform', 'distance'],
    'model__metric': ['euclidean', 'manhattan']
}

# 2. Configura a busca aleatória (O processo do Loop Interno)
# Aqui entram as 60 tentativas
random_search = RandomizedSearchCV(
    estimator=seu_modelo, 
    param_distributions=grade_de_parametros, 
    n_iter=60, 
    cv=inner_cv,
    n_jobs=-1 # Usa todos os núcleos do PC para agilizar
)

# 3. Executa o Loop Externo (A Nested CV propriamente dita)
# O cross_val_score vai rodar o random_search 10 vezes
nested_scores = cross_val_score(random_search, X, y, cv=outer_cv)

# 4. Resultado para a Estatística
print(f"Scores para ANOVA: {nested_scores}")

# <font color='lightgreen'> 🧪 ***Validação Estatística***

* ***1. Teste de Normalidade (Shapiro-Wilk):*** Antes de comparar as médias, você deve verificar se a distribuição dos resultados (os scores de cada fold do Loop Externo da Nested CV) é normal.
*       Ação: Aplique o teste de shapiro() em cada array de scores (RF, SVM, XGBoost).
*       Decisão:
*           p > 0.05: A amostra é Normal (use testes paramétricos como ANOVA).
*           p < 0.05: A amostra não é Normal (use testes não-paramétricos como Friedman). 

* ***2. Variância (Levene):*** Para saber se a oscilação entre os folds é parecida em todos os modelos.
* ***3. Comparação entre os Modelos***: Agora você testa se existe uma diferença estatisticamente significativa entre os desempenhos.
*       Cenário Normal: Execute a ANOVA de uma via para comparar as médias dos 3 modelos.
*       Cenário Não-Normal: Execute o Teste de Friedman.
*       Resultado: Se o p-valor global for menor que 0.05, significa que pelo menos um modelo é diferente dos outros
* ***4. Identificação do Melhor Modelo (Post-Hoc):*** Se o passo anterior confirmou que há diferença, você precisa identificar qual modelo é o superior.
*       Cenário Normal: Use o Teste de Tukey (HSD) para comparar os modelos par a par.
*       Cenário Não-Normal: Use o Teste de Nemenyi.
*       Escolha: O modelo com a maior média de performance que seja estatisticamente diferente dos demais será o seu vencedor oficial.
* ***5. Cálculo do Intervalo de Confiança (IC):*** Para o modelo vencedor, você calcula o IC para expressar a incerteza e a confiabilidade do resultado final.
*       Ação: Use a média e o desvio padrão dos scores do loop externo.
*       Cálculo: Geralmente utiliza-se um nível de confiança de 95%.
*       Interpretação: Isso dirá, por exemplo, que a acurácia esperada do modelo é de \(92\% \pm 3\%\).

# <font color='lightgreen'> ✅ ***Modelo Final***

Treino com os dados totais e salvamento do modelo para produção

# <font color='lightgreen'>🎯 ***Conclusão e Insights***

* Seleciona os melhores insights para o negócio
* Entregue os resultados do melhor modelo